In [59]:
import pandas as pd
import glob
import os
import numpy as np
path = r"C:\Users\23035\OneDrive\Desktop\Internship\dataset"
files = glob.glob(os.path.join(path, "*.csv"))
dfs = []
for f in files:
    data_month=pd.read_csv(f)
    date_ym=pd.to_datetime(os.path.basename(f).split('.')[0][-6:],format="%Y%m")
    data_month['date_ym']=date_ym
    dfs.append(data_month)
dataset_total_last6 = pd.concat(dfs)

C:\Users\23035\AppData\Local\Temp\ipykernel_60196\3523058081.py:9: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  data_month=pd.read_csv(f)


### 1. Handle missing values and invalid values

In [60]:
# drop the feature which miss more than 10 % values
dataset_total_last6_drop= dataset_total_last6.loc[:,dataset_total_last6.isna().mean() < 0.1]
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 124404 entries, 0 to 23259
Data columns (total 45 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   BuyerAgentAOR             118347 non-null  object        
 1   ListAgentAOR              124358 non-null  object        
 2   ViewYN                    112237 non-null  object        
 3   OriginalListPrice         124042 non-null  float64       
 4   ListingKey                124404 non-null  int64         
 5   ListAgentEmail            124134 non-null  object        
 6   CloseDate                 124404 non-null  object        
 7   ClosePrice                124403 non-null  float64       
 8   ListAgentFirstName        123903 non-null  object        
 9   ListAgentLastName         124400 non-null  object        
 10  Latitude                  124378 non-null  float64       
 11  Longitude                 124378 non-null  float64       
 12  Unparsed

In [61]:
# drop future features and unrelated features
# ClosePrice (label)
# We should use the property's features 
# It should be worked on the property whether it is listed or not
dataset_total_last6_drop = dataset_total_last6_drop.drop(columns=[
    # future features (leakage): like the feature related with list, buyer
    "ListAgentAOR",
    "BuyerAgentAOR",# leakage
    "OriginalListPrice",# leakage
    "CloseDate",# future
    "PurchaseContractDate",# future
    "ContractStatusChangeDate",# future
    "DaysOnMarket",# future
    "ListPrice",
    "ListingContractDate",

    # unrelated features
    "ListAgentEmail",
    "ListingKey",
    "ListAgentFirstName",
    "ListAgentLastName",
    "BuyerAgentMlsId",
    "ListAgentEmail",
    "ListingKeyNumeric",
    "UnparsedAddress",
    "ListOfficeName",
    "BuyerOfficeName",
    "ListAgentFullName",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "ListingKeyNumeric",
    "MlsStatus",# all Closed
    "BuyerOfficeAOR",
    "ListingId"
])


In [62]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 124404 entries, 0 to 23259
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ViewYN                 112237 non-null  object        
 1   ClosePrice             124403 non-null  float64       
 2   Latitude               124378 non-null  float64       
 3   Longitude              124378 non-null  float64       
 4   PropertyType           124404 non-null  object        
 5   LivingArea             115644 non-null  float64       
 6   CountyOrParish         124404 non-null  object        
 7   ParkingTotal           120723 non-null  float64       
 8   PropertySubType        115219 non-null  object        
 9   LotSizeAcres           113715 non-null  float64       
 10  YearBuilt              119120 non-null  float64       
 11  StreetNumberNumeric    124091 non-null  float64       
 12  BathroomsTotalInteger  118729 non-null  float64   

In [63]:
# replace unreasonable value
dataset_total_last6_drop["ClosePrice"] = dataset_total_last6_drop["ClosePrice"].replace(0, np.nan)
dataset_total_last6_drop["LotSizeSquareFeet"] = dataset_total_last6_drop["LotSizeSquareFeet"].replace(0, np.nan)
dataset_total_last6_drop["LotSizeAcres"] = dataset_total_last6_drop["LotSizeAcres"].replace(0, np.nan)
dataset_total_last6_drop["LivingArea"] = dataset_total_last6_drop["LotSizeSquareFeet"].replace(0, np.nan)

In [64]:
# remove duplicate rows
data = dataset_total_last6_drop.drop_duplicates()

In [65]:
# replace some missing value with unknown so that we will not lose too moch important data 
cols = ["PropertySubType","City","ViewYN","CountyOrParish","StateOrProvince","FireplaceYN"]
dataset_total_last6_drop[cols] = dataset_total_last6_drop[cols].fillna("Unknown")

In [66]:
# the correlation between missing value rows
missing = dataset_total_last6_drop.isna().astype(int)
missing_corr = missing.corr()
missing_corr

,ViewYN,ClosePrice,Latitude,Longitude,PropertyType,LivingArea,CountyOrParish,ParkingTotal,PropertySubType,LotSizeAcres,...,StreetNumberNumeric,BathroomsTotalInteger,City,BedroomsTotal,StateOrProvince,FireplaceYN,LotSizeArea,PostalCode,LotSizeSquareFeet,date_ym
ViewYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ClosePrice,NaN,1.000000,-0.000130,-0.000130,NaN,-0.000079,NaN,-0.001566,NaN,-0.000160,...,-0.000450,-0.001960,NaN,-0.002381,NaN,NaN,0.000571,-0.000146,-0.000079,NaN
Latitude,NaN,-0.000130,1.000000,1.000000,NaN,0.006103,NaN,-0.002525,NaN,0.005896,...,0.132477,-0.003161,NaN,0.016335,NaN,NaN,0.007794,-0.000236,0.006103,NaN
Longitude,NaN,-0.000130,1.000000,1.000000,NaN,0.006103,NaN,-0.002525,NaN,0.005896,...,0.132477,-0.003161,NaN,0.016335,NaN,NaN,0.007794,-0.000236,0.006103,NaN
PropertyType,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LivingArea,NaN,-0.000079,0.006103,0.006103,NaN,1.000000,NaN,-0.052184,NaN,0.983519,...,0.014733,-0.067727,NaN,-0.062388,NaN,NaN,0.886353,-0.003883,1.000000,NaN
CountyOrParish,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ParkingTotal,NaN,-0.001566,-0.002525,-0.002525,NaN,-0.052184,NaN,1.000000,NaN,-0.053172,...,-0.002141,0.798472,NaN,0.657323,NaN,NaN,-0.045113,-0.002844,-0.052184,NaN
PropertySubType,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LotSizeAcres,NaN,-0.000160,0.005896,0.005896,NaN,0.983519,NaN,-0.053172,NaN,1.000000,...,0.015137,-0.068936,NaN,-0.063039,NaN,NaN,0.872968,-0.003987,0.983519,NaN


In [67]:
# drop nan values
dataset_total_last6_drop=dataset_total_last6_drop.dropna()
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 102373 entries, 0 to 23256
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ViewYN                 102373 non-null  object        
 1   ClosePrice             102373 non-null  float64       
 2   Latitude               102373 non-null  float64       
 3   Longitude              102373 non-null  float64       
 4   PropertyType           102373 non-null  object        
 5   LivingArea             102373 non-null  float64       
 6   CountyOrParish         102373 non-null  object        
 7   ParkingTotal           102373 non-null  float64       
 8   PropertySubType        102373 non-null  object        
 9   LotSizeAcres           102373 non-null  float64       
 10  YearBuilt              102373 non-null  float64       
 11  StreetNumberNumeric    102373 non-null  float64       
 12  BathroomsTotalInteger  102373 non-null  float64   

### 2. Convert datatype

In [68]:
dataset_total_last6_drop["FireplaceYN"].value_counts()

FireplaceYN
True       63730
False      37118
Unknown     1525
Name: count, dtype: int64

In [69]:
dataset_total_last6_drop["ViewYN"].value_counts()

ViewYN
True       59693
False      35181
Unknown     7499
Name: count, dtype: int64

In [70]:
# yes or no
tf_map = {True: 1, False: 0, "Unknown":2}
dataset_total_last6_drop["ViewYN"]=dataset_total_last6_drop["ViewYN"].map(tf_map)
dataset_total_last6_drop["FireplaceYN"]=dataset_total_last6_drop["FireplaceYN"].map(tf_map)

In [71]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 102373 entries, 0 to 23256
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ViewYN                 102373 non-null  int64         
 1   ClosePrice             102373 non-null  float64       
 2   Latitude               102373 non-null  float64       
 3   Longitude              102373 non-null  float64       
 4   PropertyType           102373 non-null  object        
 5   LivingArea             102373 non-null  float64       
 6   CountyOrParish         102373 non-null  object        
 7   ParkingTotal           102373 non-null  float64       
 8   PropertySubType        102373 non-null  object        
 9   LotSizeAcres           102373 non-null  float64       
 10  YearBuilt              102373 non-null  float64       
 11  StreetNumberNumeric    102373 non-null  float64       
 12  BathroomsTotalInteger  102373 non-null  float64   

In [72]:
print(dataset_total_last6_drop["PropertyType"].value_counts())
dataset_total_last6_drop = pd.get_dummies( dataset_total_last6_drop, columns=["PropertyType"], drop_first=True)

PropertyType
Residential           74717
ResidentialLease      26024
ManufacturedInPark      934
ResidentialIncome       698
Name: count, dtype: int64


In [73]:
print(dataset_total_last6_drop["PropertySubType"].value_counts())
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
dataset_total_last6_drop["PropertySubType"] = le.fit_transform(dataset_total_last6_drop["PropertySubType"].astype(str))

PropertySubType
SingleFamilyResidence    72644
Condominium              15270
Townhouse                 5284
Apartment                 2769
Unknown                   2143
Duplex                    1581
ManufacturedOnLand        1129
Triplex                    464
Quadruplex                 367
StockCooperative           235
Cabin                      115
MixedUse                    94
Studio                      89
RoomingHouse                87
MobileHome                  34
OwnYourOwn                  17
Loft                        16
BoatSlip                    14
ManufacturedHome            13
CoOwnership                  3
Farm                         2
DeededParking                2
Timeshare                    1
Name: count, dtype: int64


In [74]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 102373 entries, 0 to 23256
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   ViewYN                          102373 non-null  int64         
 1   ClosePrice                      102373 non-null  float64       
 2   Latitude                        102373 non-null  float64       
 3   Longitude                       102373 non-null  float64       
 4   LivingArea                      102373 non-null  float64       
 5   CountyOrParish                  102373 non-null  object        
 6   ParkingTotal                    102373 non-null  float64       
 7   PropertySubType                 102373 non-null  int64         
 8   LotSizeAcres                    102373 non-null  float64       
 9   YearBuilt                       102373 non-null  float64       
 10  StreetNumberNumeric             102373 non-null  float64      

In [75]:
dataset_total_last6_drop.to_csv("cleaned_dataset.csv", index=False)

# 3. Test train split based on X monthes

— Training set: all months except the most recent complete month.

— Validation set: carve out the second-most-recent complete month from the training set for hyperparameter tuning and model selection — never tune on the test set.

— Test set: the last complete month of available data, touched only once, at the end.

In [76]:
X=2 # past 2 month
months = sorted(dataset_total_last6["date_ym"].unique())
data_train=dataset_total_last6[dataset_total_last6['date_ym'].isin(months[-X:-1])]
data_test=dataset_total_last6[dataset_total_last6['date_ym']==months[-1]]

In [77]:
print(data_train['date_ym'].value_counts())
print(data_test['date_ym'].value_counts())

date_ym
2026-04-01    23412
Name: count, dtype: int64
date_ym
2026-05-01    23260
Name: count, dtype: int64
